# 前言：在写注意力之前, 你需要先会这几件事

这一节主要是把后面章节会反复出现的基础组件先讲清楚，避免你在看 Self-Attention 时被训练代码分散注意力。

我们下面主要用 `torch` 包和其里面的 `torch.nn` 模块, 所以先导入.

In [1]:
import torch
from torch import nn

## 表达组件

首先介绍Transformer中基础的表达组件, 从最基础的张量开始, 到线性层, 嵌入层, 前馈神经网络FFN, Softmax.

它们是决定模型怎么表示和变换信息的基础组件.


### 张量 Tensor

可以暂时把torch里的张量理解为数学中的多维矩阵.

后面所有模块本质上都在做矩阵运算.

目前至少要先熟悉这 4 个关键词: 

- `shape`: 张量的维度结构, 比如 `torch.Size([2, 5])` 形状为2*5的张量
- `dtype`: 数据类型, 比如 `torch.float32` 32位浮点数, `torch.long/torch.int64` 64位有符号整数
- `device`: 张量所在设备, 比如 `cpu` 上计算, `cuda` 英伟达显卡上计算
- `dim`: 操作时指定的轴, 比如 `max(dim=-1)` 在最后一维取最大值

下面是张量的创建和访问属性: 

In [2]:
# shape / dtype / device
x_ids = torch.tensor([[3, 1, 8, 2, 5], [4, 7, 0, 9, 6]]) # 创建一个形状为(2, 5)的整数张量
print("x_ids =", x_ids)
print("shape =", x_ids.shape)
print("dtype =", x_ids.dtype)
print("device =", x_ids.device)
print("max = ", x_ids.max(dim=-1)[0]) # 计算每行(-1表示最后一维, 即每行)的最大值

x_ids = tensor([[3, 1, 8, 2, 5],
        [4, 7, 0, 9, 6]])
shape = torch.Size([2, 5])
dtype = torch.int64
device = cpu
max =  tensor([8, 9])


### 全连接层

最基础的线性变换层, torch中的函数是 `nn.Linear`, 数学形式上可以写成: 

$$y = xW^T + b$$

> 我们线性代数上常见的公式是 $y=Wx+b$, 这里通常默认 $x$ 是列向量, 由于在深度学习代码中默认每一行是一个样本, 这时等价写法会变成上面的公式.

在后续 Self-Attention 章节里, 信息会经过三组不同的线性层, 
分别得到 Q、K、V。现在先理解 `Linear` 的输入和输出维度就够了:  

- 如果输入张量的最后一维是 input_dim
- 线性层定义为 Linear(input_dim, output_dim)
- 那么输出最后一维会变成 output_dim

下面是用法: 

In [3]:
# 首先要创建线性层就要确定输入和输出维度, 这里设置输入维度为3, 输出维度为2
linear3x2 = nn.Linear(3, 2)
# 创建一个输入张量, 形状为(2, 3), 表示2个样本, 特征维度为3
input_tensor = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
# 将输入张量通过线性层进行矩阵乘法, 得到输出张量, 输出张量的形状为(2, 2), 表示2个样本, 特征维度为2
output_tensor = linear3x2(input_tensor)
print("Output: ", output_tensor)


Output:  tensor([[-0.6877,  0.1990],
        [-2.9541, -0.4130]], grad_fn=<AddmmBackward0>)


### 嵌入层

torch的函数是 `nn.Embedding`, 可以理解为一个可训练的查找表.

输入是整数id (token id), 输出是对应的可学习向量.

在后续章节里，我们会处理各种序列(数字序列, 中文句子等).
这些输入在进入模型前, 都会先变成 id 序列, 再交给 `Embedding` 做向量化.

数学上可以这样理解: 

$$y = E[x]$$

其中: 
- `E` 是 embedding 矩阵，形状是 `(vocab_size, d_model)`
- `x` 是 token id, 每个元素是 `0 ~ vocab_size-1` 的整数
- `E[x]` 表示按 id 从 `E` 中取对应行

如果用 one-hot 的角度看，也可以写成: 

$$y = x_{onehot} E$$

这和前面 `nn.Linear` 的关系很像: 
- `Linear` 是先乘矩阵再得到输出
- `Embedding` 是先按索引取行, 本质上等价于 one-hot 乘矩阵

现在先理解 `Embedding` 的输入和输出维度就够了: 

- 如果输入 `x` 的形状是 `(batch, seq_len)` x 中每个位置的 id 都必须在 [0, vocab_size-1]
- 层定义为 `nn.Embedding(vocab_size, d_model)` 输出是预设的d_model维度
- 那么输出形状就是 `(batch, seq_len, d_model)`

> 再强调一个初学者高频点: 输入给 `Embedding` 的 `x.dtype` 必须是整数类型, 通常用 `torch.long`(等价于 `torch.int64`).

这正是后面注意力模块的标准输入形状.

In [4]:
# 创建一个嵌入层, num_embeddings=10表示词汇表大小(vocab_size)为10, embedding_dim=3表示每个词的嵌入维度为3
embedding = nn.Embedding(num_embeddings=10, embedding_dim=3)

# 输入张量shape: (2, 5), 其中每个数字都在[0, vocab_size-1=9]之中, dtype必须是long/int64
x = torch.tensor([[3, 1, 8, 2, 5], [4, 7, 0, 9, 6]], dtype=torch.long)

# 从内置的embedding层中获取对应id的嵌入向量, 输出张量shape: (2, 5, 3)
y = embedding(x)

print("x shape:", x.shape, "dtype:", x.dtype)
print("y shape:", y.shape, "dtype:", y.dtype)
print("E shape:", embedding.weight.shape)

x shape: torch.Size([2, 5]) dtype: torch.int64
y shape: torch.Size([2, 5, 3]) dtype: torch.float32
E shape: torch.Size([10, 3])


### 前馈神经网络

前馈神经网络, Feed-Forward Network. 

简单说就是将输入特征先升维, 经过非线性的激活函数后在降维.

其中升维是为了提供更大的特征组合空间, 激活函数提供非线性表达能力, 降维是为了回到主干维度继续和后续层对接.

结构如下:

```markdown
输入 x
  -> Linear(input_dim -> ffn_dim)
  -> 激活函数(ReLU等)
  -> Linear(ffn_dim -> output_dim)
  -> 输出 y
```




In [5]:
linear3x9 = nn.Linear(3, 9) # 创建一个线性层, 输入维度为3, 输出维度为9, 线性层会自动初始化权重和偏置参数
relu = nn.ReLU() # 创建一个ReLU激活函数
linear9x3 = nn.Linear(9, 3) # 将输入维度为9的张量降维为3

x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]) # 创建一个形状为(2, 3)的整数张量
print("Input: ", x)
x = linear3x9(x) # 线性层将输入张量从形状(2, 3)转换为形状(2, 9)
print("After linear3x9: ", x)
x = relu(x) # 将线性层的输出张量通过ReLU激活函数进行非线性变换, 输出张量的形状仍然是(2, 9)
print("After ReLU: ", x)
x = linear9x3(x) # 将输入张量从形状(2, 9)转换为形状(2, 3)
print("Output: ", x)



Input:  tensor([[1., 2., 3.],
        [4., 5., 6.]])
After linear3x9:  tensor([[ 0.0754, -0.6354, -0.2778, -1.6876, -0.8403, -1.8603, -1.9725, -0.0285,
         -0.6061],
        [-0.3357, -1.2376,  0.2424, -3.1510, -3.0253, -3.9316, -4.1045, -0.6465,
         -1.2458]], grad_fn=<AddmmBackward0>)
After ReLU:  tensor([[0.0754, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.2424, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<ReluBackward0>)
Output:  tensor([[ 0.1248,  0.0011,  0.0385],
        [ 0.1869, -0.0183, -0.0581]], grad_fn=<AddmmBackward0>)


### softmax

在后续注意力章节中, `softmax` 函数会非常高频地出现.

由于该函数能将处在实数域的输入元素映射到在(0, 1)范围的输出元素, 并且所有输出元素相加等于1, 所以非常适合用来把一组实数分数(logits)转换为概率分布.

数学形式: 

$$\mathrm{softmax}(z_i)=\frac{e^{z_i}}{\sum_j e^{z_j}}$$

你可以先记住 3 个结论: 

- 输出范围在 `(0, 1)`
- 同一维度上的所有输出之和等于 `1`
- 输入分数越大，对应位置的概率越大

在 torch 里的写法是: 

`torch.softmax(logits, dim=-1)` 或者先

`import torch.nn.functional as F`, 再 `F.softmax(logits, dim=-1)` 调用函数, 两者一样.

为了代码简洁统一, 我们统一用前者.

这里的 `dim` 很重要: 
- `dim=-1` 表示在最后一维做归一化(最常用)
- 如果 logits 形状是 `(batch, seq_len, seq_len)`, 通常也在最后一维做 `softmax`

> 数值稳定性说明(了解即可): 实际实现通常会先减去最大值再做指数运算，避免进行 `exp` 运算时数字太大溢出. 也就是 $softmax(z) = softmax(z - max(z))$.

In [6]:
# 创建一个logits张量, 形状为(2, 3), 表示2个样本, 每个样本有3个类别的logits值
logits = torch.tensor([[2.0, 1.0, 0.1],
                       [0.5, 2.5, 1.0]])

# 计算softmax概率, dim=-1表示在最后一个维度上进行softmax归一化, 也就是对每行进行归一化
prob = torch.softmax(logits, dim=-1)

print("logits =", logits)
print("softmax =", prob)
print("row sum =", prob.sum(dim=-1)) # 验证每行的softmax概率和为1

logits = tensor([[2.0000, 1.0000, 0.1000],
        [0.5000, 2.5000, 1.0000]])
softmax = tensor([[0.6590, 0.2424, 0.0986],
        [0.0996, 0.7361, 0.1643]])
row sum = tensor([1.0000, 1.0000])


## 训练增强技巧

这部分主要介绍用于提升训练稳定性的技巧, 用来缓解深层网络中常见的梯度消失, 梯度爆炸和过拟合等问题. 由于 Transformer 往往需要堆叠多层, 参数规模较大, 训练时更容易出现这些现象, 因此需要借助这些组件来提高优化稳定性和泛化能力.

### 归一化

归一化的作用是将特征调整到更稳定的尺度, 减少不同层之间的数值波动, 从而让模型更容易训练. 它可以缓解梯度消失, 梯度爆炸以及训练不稳定等问题. 常见的归一化方法包括批归一化和层归一化. 它们都可以理解为一种标准化操作, 核心思想是减去均值, 再除以标准差, 但具体统计范围不同.

标准化公式为:

$$y=\frac{x-\mu}{\sigma}$$

批归一化是对一个张量的每个特征维度分别做标准化, 层归一化是每个样本内部的特征维度做归一化

In [ ]:
# 创建一个形状为(2, 3)的整数张量, 我们把第一个维度看作batch_size, 第二个维度看作特征维度, 也就是有两批样本, 每批有三个特征
x = torch.tensor([[1.0, 2.0, 3.0], 
                  [4.0, 5.0, 6.0]])

# 批归一化是对每个特征维度进行归一化, 也就是对每列进行归一化, 计算每列的均值和标准差, 然后对每个元素进行归一化变换
batch_norm = nn.BatchNorm1d(num_features=3) # 创建一个批归一化层, 输入特征维度为3
batch_output = batch_norm(x) # 将输入张量通过批归一化层进行归一化变换, 输出张量的形状仍然是(2, 3)
print("BatchNorm output: ", batch_output)

# 层归一化是对每个样本进行归一化, 也就是对每行进行归一化, 计算每行的均值和标准差, 然后对每个元素进行归一化变换
layer_norm = nn.LayerNorm(normalized_shape=3) # 创建一个层归一化层, normalized_shape=3表示对最后一个维度进行归一化
layer_output = layer_norm(x) # 将输入张量通过层归一化层进行
print("LayerNorm output: ", layer_output)



BatchNorm output:  tensor([[-1.0000, -1.0000, -1.0000],
        [ 1.0000,  1.0000,  1.0000]], grad_fn=<NativeBatchNormBackward0>)
LayerNorm output:  tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]], grad_fn=<NativeLayerNormBackward0>)


### 残差连接

在深层模型中, 信息需要经过多层连续变换, 训练时容易出现梯度消失和梯度爆炸. 如果某一层学得不好, 输出可能接近于零, 后续层的输入就会被削弱, 信息也更容易在深层传递中丢失.

残差连接的作用是把当前层的变换结果直接与原输入相加, 即输出: 

$$y=x+F(x)$$

这样一来, 即使某一层暂时没有学到有效特征, 网络也至少可以保留原始信息, 同时让梯度拥有更顺畅的传播路径, 从而提高深层网络的可训练性. 虽然简单, 但是好用.

In [8]:
def f(x): # 定义一个函数f, 模拟指数衰减函数, 输入x是一个张量, 输出是x乘以0.5, 每次迭代都会将输入张量的值减半
    return x * 0.5

x = torch.tensor([[1.0, 2.0, 3.0], 
                  [4.0, 5.0, 6.0]])

# 无残差连接, 迭代10次应用函数f, 每次将上一次的输出作为下一次的输入
x1 = x.clone() # 克隆输入张量, 避免修改原始输入
for _ in range(10):
    x1 = f(x1)
print("Output without residual connection: ", x1)

# 有残差连接, 迭代10次应用函数f, 每次将上一次的输出加上输入张量x作为下一次的输入
x2 = x.clone()
for _ in range(10):
    x2 = f(x2) + x2
print("Output with residual connection: ", x2)


Output without residual connection:  tensor([[0.0010, 0.0020, 0.0029],
        [0.0039, 0.0049, 0.0059]])
Output with residual connection:  tensor([[ 57.6650, 115.3301, 172.9951],
        [230.6602, 288.3252, 345.9902]])


### Dropout

Dropout的作用是将一部分训练参数置零, 迫使模型学到更稳健的模式.

In [9]:
x = torch.tensor([[1.0, 2.0, 3.0],
                [4.0, 5.0, 6.0]]) # 创建一个形状为(2, 3)的张量

dropout = nn.Dropout(0.5) # 创建一个Dropout层, p=0.5表示在训练过程中以50%的概率将输入张量的元素随机置零, 以防止过拟合

dropout_output = dropout(x) # 将输入张量通过Dropout层进行随机置零, 输出张量的形状仍然是(2, 3)
print("Dropout output: ", dropout_output)

Dropout output:  tensor([[ 2.,  4.,  6.],
        [ 0., 10., 12.]])


## 训练流程

整个训练流程是这样的: 
1. 首先通过预先设计的神经网络模型 `model` 前向计算给出预测值, 
2. 其次通过合适的损失函数 `criterion` 计算真实值与预测值的差距, 得到损失值 `loss`, 
3. 然后通过根据损失值 `loss` 进行反向传播算法 `backward` , 按链式法则计算每个可训练参数的梯度,
4. 最后优化器 `optimizer` 根据这些梯度更新参数, 从而让模型在下一轮训练中得到更小的损失.

可以这样记: 
- `model`: 前向计算, 负责给出预测
- `criterion`: 衡量预测与标签差距, 得到损失值loss
- `backward`: 根据loss进行反向传播计算梯度
- `optimizer`: 根据梯度更新参数, 让下次预测更好

## 最小训练循环

下面是用torch实现的最小训练循环示意: 

```python

for i in range(epochs):
    optimizer.zero_grad()           # 1) 清空上一轮累计梯度
    pred = model(train_x)           # 2) 前向计算得到预测
    loss = criterion(pred, train_y) # 3) 计算损失
    loss.backward()                 # 4) 反向传播, 计算每个参数的梯度
    optimizer.step()                # 5) 用梯度更新参数
```

### Q&A

为什么要循环epochs次?

为了充分利用训练数据训练模型.

为什么每轮都要 `zero_grad()`?

因为 PyTorch 默认会累加梯度, 不清空就会把历史梯度叠加到当前轮, 训练行为会失真.

为什么 `backward()` 在 `step()` 前?

因为优化器更新参数需要先拿到梯度, 而梯度是由 `backward()` 计算出来的.


## 初学者常见坑（建议先避开）

1. 输入给 Embedding 的张量类型必须是 long.
2. 关注维度变化, 确保进行矩阵乘法的张量维度正确, 尤其是 batch / seq_len / hidden_dim 的位置.
3. 先保证一小批数据能跑通, 再扩大 batch 和 epochs.
4. loss 不下降时, 先检查标签格式和输出维度是否匹配.

如果你准备好了, 就可以进入第一章, 开始实现最小可用的 Self-Attention.